# Regression by Minimizing Absolute Deviation — Complete Workflow

This notebook reproduces the repository workflow from data loading to final validation.

**Empirical datasets:** Boston Housing, Concrete Compressive Strength, Hawkins–Bradu–Kass (HBK).

**Core literature:** Wagner (1959), Barrodale & Roberts (1973), Bassett & Koenker (1978), Bloomfield & Steiger (1980), Narula & Wellington (1982), and Pollard (1991).

## Objective

> How does regression obtained by minimizing absolute deviations differ from regression obtained by minimizing squared deviations, particularly when observations contain large or extreme errors?


## 1. Mathematical formulation

For observation \(i\),

$$
r_i = y_i - x_i^T\beta.
$$

Ordinary Least Squares minimizes

$$
\min_{\beta}\sum_{i=1}^{n} r_i^2.
$$

Least Absolute Deviations minimizes

$$
\min_{\beta}\sum_{i=1}^{n}|r_i|.
$$

For the linear-programming form of LAD,

$$
r_i=e_i^+-e_i^-,
\qquad
e_i^+\ge 0,\quad e_i^-\ge 0,
$$

and the objective is

$$
\min \sum_{i=1}^{n}\left(e_i^+ + e_i^-\right).
$$


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Image, display

from absolute_deviation.data import DATASETS, load_dataset
from absolute_deviation.experiments import (
    RESULT_DIR,
    FIGURE_DIR,
    run_original_data,
    run_contamination_experiment,
    run_error_distribution_experiment,
    run_runtime_benchmark,
    validate_lad_solver,
)
from absolute_deviation.models import fit_ols, fit_lad
from absolute_deviation.plotting import (
    plot_clean_and_contaminated_lines,
    plot_contamination_results,
    plot_distribution_results,
    plot_runtime_results,
    plot_real_dataset_predictions,
    plot_hbk_multivariate_inlier_outlier,
    plot_unlabeled_multivariate_residual_space,
)

pd.set_option("display.max_columns", 50)
ROOT

## 2. Load and inspect the empirical datasets

In [ ]:
dataset_rows = []
for name in DATASETS:
    X, y, columns = load_dataset(name)
    dataset_rows.append({
        "dataset": name,
        "observations": len(y),
        "predictors": X.shape[1],
        "target": DATASETS[name]["target"],
        "predictor_names": ", ".join(columns),
    })

dataset_summary = pd.DataFrame(dataset_rows)
display(dataset_summary)

## 3. Fit OLS and LAD on the original datasets

The same rows and predictors are used for both models. OLS minimizes SSE and LAD minimizes SAE on the fitted sample.


In [ ]:
original_metrics, original_coefficients = run_original_data()
display(original_metrics)
display(original_coefficients.head(20))

## 4. Empirical multivariate visualizations

In [ ]:
plot_real_dataset_predictions()
plot_hbk_multivariate_inlier_outlier()
plot_unlabeled_multivariate_residual_space("boston_housing", "Boston Housing")
plot_unlabeled_multivariate_residual_space("concrete_strength", "Concrete Strength")

for filename in [
    "hbk_multivariate_inlier_outlier.png",
    "boston_housing_multivariate_residuals.png",
    "concrete_strength_multivariate_residuals.png",
    "boston_housing_actual_vs_fitted.png",
    "concrete_strength_actual_vs_fitted.png",
    "hbk_actual_vs_fitted.png",
]:
    display(Image(filename=str(FIGURE_DIR / filename)))

## 5. Controlled large-response-error experiment

Large vertical response errors are introduced into a controlled regression design. The contamination fractions are project-selected illustration settings.


In [ ]:
contamination_metrics, contamination_shifts = run_contamination_experiment()
plot_clean_and_contaminated_lines()
plot_contamination_results()

contamination_summary = (
    contamination_metrics
    .groupby(["contamination_fraction", "model"])[["MAE", "RMSE"]]
    .mean()
    .reset_index()
)
shift_summary = (
    contamination_shifts
    .groupby(["contamination_fraction", "model"])[["coefficient_shift_l2", "prediction_shift_mae"]]
    .mean()
    .reset_index()
)

display(contamination_summary)
display(shift_summary)

for filename in [
    "ols_lad_clean_fit.png",
    "ols_lad_contaminated_fit.png",
    "mae_vs_contamination.png",
    "rmse_vs_contamination.png",
    "coefficient_shift_vs_contamination.png",
]:
    display(Image(filename=str(FIGURE_DIR / filename)))

## 6. Error-distribution experiment

OLS and LAD are compared under Normal, Laplace, Cauchy, and contaminated-normal errors.


In [ ]:
distribution_results = run_error_distribution_experiment()
plot_distribution_results()

distribution_summary = (
    distribution_results
    .groupby(["distribution", "model"])[["coefficient_error_l2", "SSE", "SAE", "MAE", "RMSE"]]
    .median()
    .reset_index()
)

display(distribution_summary)
display(Image(filename=str(FIGURE_DIR / "error_distribution_comparison.png")))

## 7. Computational comparison

In [ ]:
runtime_results = run_runtime_benchmark()
plot_runtime_results()

runtime_summary = (
    runtime_results
    .groupby(["n", "p", "model"])["runtime_seconds"]
    .median()
    .reset_index()
)

display(runtime_summary)
display(Image(filename=str(FIGURE_DIR / "runtime_vs_sample_size.png")))

## 8. Solver validation

For deterministic examples, the OLS solution must have SSE no larger than the SSE at the LAD coefficients, and the LAD solution must have SAE no larger than the SAE at the OLS coefficients.


In [ ]:
validation = validate_lad_solver()
display(validation)

assert validation["ols_sse_is_minimum_against_lad"].all()
assert validation["lad_sae_is_minimum_against_ols"].all()

print("All OLS/LAD objective checks passed.")

## 9. Generated outputs

In [ ]:
result_files = sorted(path.name for path in RESULT_DIR.glob("*.csv"))
figure_files = sorted(path.name for path in FIGURE_DIR.glob("*.png"))

print("Result files:")
for name in result_files:
    print(" -", name)

print("\nFigure files:")
for name in figure_files:
    print(" -", name)

## 10. Conclusion

The workflow compares squared-loss and absolute-loss regression on the same empirical data and under controlled large-error and long-tailed settings. OLS directly minimizes squared residuals, while LAD directly minimizes absolute residuals. The project therefore treats their differences as consequences of the loss function and error behavior rather than claiming that either estimator is universally superior.
